In [ ]:
import csv
import json
from datetime import datetime

LIMITE_SUSPEITO = 10000.00

def formatar_moeda(valor: float) -> str:
    """Formata número no padrão brasileiro R$ 1.234,56."""
    return f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

def ler_transacoes(caminho_arquivo: str = "transacoes.csv") -> list[dict]:
    """
    Lê o CSV com DictReader detectando se o separador é ponto e vírgula ou vírgula.
    Trata FileNotFoundError caso o arquivo não exista.
    """
    try:
        with open(caminho_arquivo, mode="r", encoding="utf-8-sig") as arquivo:
            primeira_linha = arquivo.readline()
            delimitador = ";" if ";" in primeira_linha else ","
            arquivo.seek(0)

            leitor = csv.DictReader(arquivo, delimiter=delimitador)
            return list(leitor)
    except FileNotFoundError:
        print(f"❌ Erro: O arquivo '{caminho_arquivo}' não foi encontrado.")
        return []

linhas_brutas = ler_transacoes("transacoes.csv")
print(f"Total de linhas lidas: {len(linhas_brutas)}")
print(f"Chaves separadas corretamente: {list(linhas_brutas[0].keys())}")

In [ ]:
def validar_transacao(linha: dict) -> dict | None:
    """
    Valida uma linha do CSV:
    - id: inteiro válido
    - cliente_id: preenchido
    - data: AAAA-MM-DD
    - tipo: credito ou debito
    - valor: decimal > 0
    Descarte silencioso (retorna None) em caso de erro.
    """
    # 1. Validação de ID
    raw_id = linha.get("id", "").strip()
    if not raw_id or not raw_id.isdigit():
        return None
    id_transacao = int(raw_id)

    # 2. Validação do cliente_id
    cliente_id = linha.get("cliente_id", "").strip()
    if not cliente_id:
        return None

    # 3. Validação de Data (try/except)
    raw_data = linha.get("data", "").strip()
    try:
        data_obj = datetime.strptime(raw_data, "%Y-%m-%d")
    except ValueError:
        return None

    # 4. Validação do Tipo
    tipo = linha.get("tipo", "").strip().lower()
    if tipo not in ["credito", "debito"]:
        return None

    # 5. Validação do Valor (try/except)
    raw_valor = linha.get("valor", "").strip().replace(",", ".")
    try:
        valor = float(raw_valor)
        if valor <= 0:
            return None
    except ValueError:
        return None

    return {
        "id": id_transacao,
        "data_str": raw_data,
        "data": data_obj,
        "mes": data_obj.strftime("%Y-%m"),
        "cliente_id": cliente_id,
        "tipo": tipo,
        "valor": valor,
        "descricao": linha.get("descricao", "").strip(),
        "categoria": linha.get("categoria", "").strip().lower()
    }

# Processamento da limpeza
transacoes_validas = []
ids_processados = set()

for linha in linhas_brutas:
    registro = validar_transacao(linha)
    if registro is not None and registro["id"] not in ids_processados:
        ids_processados.add(registro["id"])
        transacoes_validas.append(registro)

total_lidas = len(linhas_brutas)
total_validas = len(transacoes_validas)
total_invalidas = total_lidas - total_validas

# Resumo da limpeza exigido no terminal
print(f"Total de linhas lidas: {total_lidas}")
print(f"Linhas válidas: {total_validas}")
print(f"Linhas inválidas: {total_invalidas}")

In [ ]:
def gerar_relatorio(transacoes_validas: list[dict], total_invalidas: int) -> dict:
    """
    Agrupa os dados por mês, calcula as métricas exigidas,
    calcula os dias decorridos no período e lista transações suspeitas.
    """
    if not transacoes_validas:
        return {}

    # Cálculo do período analisado (dias decorridos)
    datas = [t["data"] for t in transacoes_validas]
    data_mais_antiga = min(datas)
    data_mais_recente = max(datas)
    dias_totais = (data_mais_recente - data_mais_antiga).days

    # Obtenção dos meses únicos e ordenados
    meses = sorted(list(set(t["mes"] for t in transacoes_validas)))
    resumo_mensal = {}

    for mes in meses:
        itens_mes = [t for t in transacoes_validas if t["mes"] == mes]
        qtd = len(itens_mes)
        credito = sum(t["valor"] for t in itens_mes if t["tipo"] == "credito")
        debito = sum(t["valor"] for t in itens_mes if t["tipo"] == "debito")
        saldo = credito - debito
        valores = [t["valor"] for t in itens_mes]
        media = sum(valores) / qtd if qtd > 0 else 0.0
        maior_valor = max(valores) if valores else 0.0
        menor_valor = min(valores) if valores else 0.0

        resumo_mensal[mes] = {
            "quantidade": qtd,
            "total_credito": round(credito, 2),
            "total_debito": round(debito, 2),
            "saldo": round(saldo, 2),
            "media": round(media, 2),
            "maior_valor": round(maior_valor, 2),
            "menor_valor": round(menor_valor, 2)
        }

    # Sinalização de transações suspeitas
    suspeitas = [
        {
            "id": t["id"],
            "cliente_id": t["cliente_id"],
            "data": t["data_str"],
            "valor": round(t["valor"], 2)
        }
        for t in transacoes_validas if t["valor"] > LIMITE_SUSPEITO
    ]

    return {
        "gerado_em": datetime.now().strftime("%Y-%m-%d"),
        "total_transacoes_validas": len(transacoes_validas),
        "total_transacoes_invalidas": total_invalidas,
        "periodo": {
            "data_inicio": data_mais_antiga.strftime("%Y-%m-%d"),
            "data_fim": data_mais_recente.strftime("%Y-%m-%d"),
            "dias_totais": dias_totais
        },
        "resumo_mensal": resumo_mensal,
        "transacoes_suspeitas": suspeitas
    }

# --- Teste da Célula 3 no terminal ---
relatorio_teste = gerar_relatorio(transacoes_validas, total_invalidas)

print("===== RELATÓRIO MENSAL =====")
for mes, metricas in relatorio_teste["resumo_mensal"].items():
    print(f"Mês: {mes}")
    print(f"  Transações:   {metricas['quantidade']}")
    print(f"  Total crédito: {formatar_moeda(metricas['total_credito'])}")
    print(f"  Total débito:  {formatar_moeda(metricas['total_debito'])}")
    print(f"  Saldo:         {formatar_moeda(metricas['saldo'])}")
    print(f"  Média:         {formatar_moeda(metricas['media'])}")
    print(f"  Maior valor:   {formatar_moeda(metricas['maior_valor'])}")
    print(f"  Menor valor:   {formatar_moeda(metricas['menor_valor'])}")

In [ ]:
def exibir_relatorio(relatorio: dict, total_lidas: int):
    """Exibe o relatório formatado no terminal com layout padronizado."""
    print("=" * 60)
    print("                  CLEARBANK - RELATÓRIO                  ")
    print("=" * 60)
    print(f"Total de linhas lidas: {total_lidas}")
    print(f"Linhas válidas:        {relatorio['total_transacoes_validas']}")
    print(f"Linhas inválidas:      {relatorio['total_transacoes_invalidas']}")

    p = relatorio["periodo"]
    print(f"Período analisado:     {p['data_inicio']} até {p['data_fim']} ({p['dias_totais']} dias decorridos)")

    print("\n===== RELATÓRIO MENSAL =====")
    for mes, metricas in relatorio["resumo_mensal"].items():
        print(f"Mês: {mes}")
        print(f"  Transações:    {metricas['quantidade']}")
        print(f"  Total crédito: {formatar_moeda(metricas['total_credito'])}")
        print(f"  Total débito:  {formatar_moeda(metricas['total_debito'])}")
        print(f"  Saldo:         {formatar_moeda(metricas['saldo'])}")
        print(f"  Média:         {formatar_moeda(metricas['media'])}")
        print(f"  Maior valor:   {formatar_moeda(metricas['maior_valor'])}")
        print(f"  Menor valor:   {formatar_moeda(metricas['menor_valor'])}\n")

    print("===== TRANSAÇÕES SUSPEITAS =====")
    if relatorio["transacoes_suspeitas"]:
        for s in relatorio["transacoes_suspeitas"]:
            print(f"ID: {s['id']} | Cliente: {s['cliente_id']} | Data: {s['data']} | Valor: {formatar_moeda(s['valor'])}")
    else:
        print("Nenhuma transação suspeita encontrada.")
    print("=" * 60)

def salvar_json(dados_relatorio: dict, caminho_arquivo: str = "relatorio.json"):
    """Exporta o resultado da análise para o arquivo relatorio.json."""
    with open(caminho_arquivo, mode="w", encoding="utf-8") as arquivo:
        json.dump(dados_relatorio, arquivo, ensure_ascii=False, indent=2)
    print(f"\n💾 Arquivo '{caminho_arquivo}' gerado com sucesso!")

In [ ]:
def main():
    arquivo_csv = "transacoes.csv"
    arquivo_saida = "relatorio.json"

    # 1. Leitura
    linhas_brutas = ler_transacoes(arquivo_csv)
    if not linhas_brutas:
        return

    total_lidas = len(linhas_brutas)

    # 2. Limpeza e descarte de duplicidades por ID
    transacoes_validas = []
    ids_registrados = set()
    for linha in linhas_brutas:
        registro = validar_transacao(linha)
        if registro is not None and registro["id"] not in ids_registrados:
            ids_registrados.add(registro["id"])
            transacoes_validas.append(registro)

    total_validas = len(transacoes_validas)
    total_invalidas = total_lidas - total_validas

    # 3. Geração de Métricas
    relatorio = gerar_relatorio(transacoes_validas, total_invalidas)

    # 4. Impressão estruturada
    exibir_relatorio(relatorio, total_lidas)

    # 5. Exportação JSON
    salvar_json(relatorio, arquivo_saida)

main()